In [1]:
import ray
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

In [2]:
if ray.is_initialized():
    ray.shutdown()
ray.init()
print(ray.cluster_resources())

2023-11-14 13:52:15,011	INFO worker.py:1633 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


{'node:127.0.0.1': 1.0, 'object_store_memory': 1777504665.0, 'memory': 3555009332.0, 'node:__internal_head__': 1.0, 'CPU': 12.0, 'GPU': 1.0}


#### Load data

In [3]:
@ray.remote(resources=None,num_cpus=1, num_gpus=0, max_calls=1, num_returns=1) #  resources={"CustomResource": 1}
def read_data(bucket="../data/input/general_info.txt"):
    return ray.data.read_text(bucket)

#### NLP

In [5]:
@ray.remote(num_cpus=1, max_calls=1, num_returns=1)
def find_dois(ds: ray.data.Dataset) -> ray.data.Dataset:
    """
    Extract DOIs from the given dataset using regex.
    """
    doi_regex = re.compile(r'\b10\.[0-9]{4,9}/[-._;():A-Za-z0-9]{1,100}\b')
    return ds.map(lambda batch: {"doi": doi_regex.findall(str(batch["text"]))})


@ray.remote(num_cpus=3, max_calls=1, num_returns=1)
def biobert_predict_diseases(ds: ray.data.Dataset, model_path="../data/biobert_diseases_ner"):
    """
    Predict diseases in the given text using a custom-loaded BioBERT model.
    """
    texts = str([record["text"] for record in ds.take()])
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer)
    dis_result = nlp_ner(texts)
    filtered_results = [entry for entry in dis_result if "DISEASE" in entry["entity"] and len(entry["word"]) > 1]
    filtered_results = ray.data.from_items(filtered_results)
    return filtered_results.map(lambda row: {"disease": row})

In [35]:
@ray.remote(num_cpus=2, max_calls=1, num_returns=1)
def write_to_json(doi_result,diseases_result):
    """
    Save combined DOIs and diseases data from Ray datasets to a JSON file.
    """
    import json
    
    doi_list = doi_result.take()
    disease_list = diseases_result.take()

    dois = [entry["doi"] for entry in doi_list if len(entry["doi"]) > 0]
    diseases = [entry["disease"] for entry in disease_list]

    combined_results = {
        "dois": dois,
        "diseases": diseases
    }
    with open("../data/output/test/combined_results.json", "w") as file:
        json.dump(combined_results, file, indent=4)
    print("Saved results to JSON file.")

#### Workflow

In [32]:
ds = ray.get(read_data.remote())

doi_result = ray.get(find_dois.remote(ds))
diseases_result = ray.get(biobert_predict_diseases.remote(ds))

ray.get(write_to_json.remote(doi_result,diseases_result))

(read_data pid=25388) To satisfy the requested parallelism of 24, each read task output is split into 24 smaller blocks.
(biobert_predict_diseases pid=15176) Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
(biobert_predict_diseases pid=15176) Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadText->SplitBlocks(24)] -> LimitOperator[limit=20]
(biobert_predict_diseases pid=15176) Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
(biobert_predict_diseases pid=15176) Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`
(biobert_predict_diseases pid=15176) Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


(pid=15176) Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

(write_to_json pid=10384) Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadText->SplitBlocks(24)] -> TaskPoolMapOperator[Map(<lambda>)] -> LimitOperator[limit=20]


(pid=10384) Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

(write_to_json pid=10384) Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[Map(<lambda>)] -> LimitOperator[limit=20]


(pid=10384) Running 0:   0%|          | 0/1 [00:00<?, ?it/s]


### Save results

In [43]:
result_path = "../data/output/test"

In [23]:
combined_ds = doi_result.union(diseases_result)

combined_ds.write_json(
    result_path+"/combined_result",
    try_create_dir=True,
)

doi_result.repartition(1).write_json(
    result_path + "/doi_result",
    try_create_dir=True
)

diseases_result.repartition(1).write_json(
    result_path + "/diseases_result",
    try_create_dir=True
)
diseases_result.repartition(1).write_parquet(
    result_path + "/diseases_result",
    try_create_dir=True
)

2023-11-10 12:02:31,398	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[ReadText->SplitBlocks(24)] -> TaskPoolMapOperator[Map(<lambda>)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[Write]
2023-11-10 12:02:31,399	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
2023-11-10 12:02:31,399	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


- Repartition 1:   0%|          | 0/1 [00:00<?, ?it/s]

Split Repartition 2:   0%|          | 0/1 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

2023-11-10 12:02:31,513	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[Map(<lambda>)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[Write]
2023-11-10 12:02:31,513	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
2023-11-10 12:02:31,514	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


...

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

2023-11-10 12:02:31,676	INFO streaming_executor.py:93 -- Executing DAG InputDataBuffer[Input] -> TaskPoolMapOperator[Map(<lambda>)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[Write]
2023-11-10 12:02:31,677	INFO streaming_executor.py:94 -- Execution config: ExecutionOptions(resource_limits=ExecutionResources(cpu=None, gpu=None, object_store_memory=None), locality_with_output=False, preserve_order=False, actor_locality_enabled=True, verbose_progress=False)
2023-11-10 12:02:31,677	INFO streaming_executor.py:96 -- Tip: For detailed progress reporting, run `ray.data.DataContext.get_current().execution_options.verbose_progress = True`


- Repartition 1:   0%|          | 0/1 [00:00<?, ?it/s]

Split Repartition 2:   0%|          | 0/1 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]